# Random Forest

Standard tabular ML workflow for LD50 regression.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
from sklearn.ensemble import RandomForestRegressor as SklearnRandomForest

MODELING_DIR = Path("../modeling").resolve()
if str(MODELING_DIR) not in sys.path:
    sys.path.append(str(MODELING_DIR))

from evaluation import save_ml_artifacts
from preprocessing import load_and_preprocess_tabular
from storage import artifact_paths


ModuleNotFoundError: No module named 'evaluation'

In [ ]:
MODEL_NAME = "random_forest"
BASE_DIR = Path.cwd()
RANDOM_STATE = 42

X_train, X_val, X_test, y_train, y_val, y_test, feature_names, preprocessor = load_and_preprocess_tabular()
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)
paths = artifact_paths(BASE_DIR, MODEL_NAME, model_suffix=".pkl", importance=True)


In [ ]:
try:
    from cuml.ensemble import RandomForestRegressor as CumlRandomForest

    model = CumlRandomForest(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        random_state=RANDOM_STATE,
    )
    use_gpu = True
except Exception:
    warnings.warn("cuML RandomForestRegressor not available; using sklearn on CPU.")
    model = SklearnRandomForest(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    use_gpu = False

model.fit(X_train, y_train)
predictions = model.predict(X_test)


In [ ]:
metrics = save_ml_artifacts(
    model_name="Random Forest",
    model=model,
    paths=paths,
    X_test=X_test,
    y_test=y_test,
    predictions=predictions,
    feature_names=feature_names,
    plot_color="orange",
)
metrics
